# jaxfne — Étude No. 12 · Continuous Omission Oddball (COOP)

A continuous, rhythmic pulse train (6 Hz) where each pulse is randomly
omitted with some probability, run as ONE continuous simulation rather
than discrete standard/omitted trials (Étude 11's design). This is the
"global" counterpart to Étude 11's single-slot local omission: the
omission-locked response here reflects violation of an ongoing rhythmic
expectation built up over several preceding pulses, not a single fixed
trial structure.

Uses `jaxfne.coop_omission_oddball_paradigm` (`jaxfne/paradigm.py`,
already implemented) directly — unlike `omission_oddball_paradigm`
(Étude 11), COOP's own omitted-pulse events set `metadata={'drive_amplitude':
0.0}` explicitly, so they are genuinely silent, not merely unlabeled
(confirmed by inspection, see F-022 in skills/FRICTIONS_STACK.md for the
contrast with `omission_oddball_paradigm`'s marker-event gap).

## 1. Setup

In [1]:
import numpy as np
import jax.numpy as jnp
import jaxfne as jtfne

print("jaxfne", jtfne.__version__)


jaxfne 0.4.4


## 2. Config

In [2]:
N_NEURONS = 200
SEED = 0
DT_MS = 0.5
DUR_MS = 2000.0        # continuous run duration per trial
FREQ_HZ = 6.0           # pulse rate
OMISSION_PROB = 0.2     # per-pulse omission probability
N_TRIALS = 8            # independent continuous runs (paradigm re-seeded per trial)

cfg = (
    jtfne.build_laminar_column(name="V1", n=N_NEURONS, ei_profile="canonical")
    .runtime(seed=SEED, recurrent_backend="edge_list")
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "V_m"], n_contacts=8)
    .field(domain="laminar_column", conductivity="proxy", boundary="mean_zero_neumann")
)
model = jtfne.construct(cfg)
nt = model.neuron_table()
l4e_idx = [i for i, r in enumerate(nt) if r.get("layer") == "L4" and r.get("cell_type") == "E"]
print(f"L4 E neurons: {len(l4e_idx)} / {N_NEURONS}")


L4 E neurons: 14 / 200


## 3. Paradigm

`coop_omission_oddball_paradigm` builds its own pulse train and omission
draws internally from `seed` — a new `seed` per trial gives an
independent pulse-train realization (different omission positions each
trial), not just a different simulation noise draw.

In [3]:
def make_coop_condition(trial_seed):
    paradigm = jtfne.coop_omission_oddball_paradigm(
        duration_ms=DUR_MS, dt_ms=DT_MS, freq_hz=FREQ_HZ,
        omission_prob=OMISSION_PROB, target_indices=l4e_idx, seed=trial_seed,
    )
    return paradigm.conditions[0]

probe_cond = make_coop_condition(SEED)
pulse_events = [e for e in probe_cond.events if e.label != "trial_start"]
print(f"pulses per trial: {len(pulse_events)}, expected omissions: ~{len(pulse_events) * OMISSION_PROB:.1f}")


pulses per trial: 10, expected omissions: ~2.0


## 4. Run

In [4]:
def event_locked_rates(sig, events):
    """Post-onset L4 E firing rate for each non-marker event, split by omission."""
    spk = np.asarray(sig.spikes)
    standard_rates, omitted_rates = [], []
    for e in events:
        window_start = int(e.onset_ms / DT_MS)
        window_end = int((e.onset_ms + e.duration_ms + 50.0) / DT_MS)
        window_end = min(window_end, spk.shape[0])
        rate = spk[window_start:window_end, l4e_idx].mean() * 1000.0 / DT_MS
        (omitted_rates if e.is_omission else standard_rates).append(rate)
    return standard_rates, omitted_rates

all_standard_rates, all_omitted_rates = [], []
for i in range(N_TRIALS):
    cond = make_coop_condition(SEED + i)
    pulse_events = [e for e in cond.events if e.label != "trial_start"]
    sig = jtfne.simulate(
        model, sim=jtfne.Simulation(duration_ms=DUR_MS, dt_ms=DT_MS, seed=SEED + i),
        paradigm=cond,
    )
    assert bool(jnp.all(jnp.isfinite(sig.V_m))), "non-finite V_m -- do not trust this run"
    std_rates, omit_rates = event_locked_rates(sig, pulse_events)
    all_standard_rates.extend(std_rates)
    all_omitted_rates.extend(omit_rates)

all_standard_rates = np.array(all_standard_rates)
all_omitted_rates = np.array(all_omitted_rates)
print(f"standard pulses: n={len(all_standard_rates)}  rate={all_standard_rates.mean():.2f}+-{all_standard_rates.std():.2f} Hz")
print(f"omitted pulses:  n={len(all_omitted_rates)}  rate={all_omitted_rates.mean():.2f}+-{all_omitted_rates.std():.2f} Hz")


standard pulses: n=62  rate=13.69+-3.80 Hz
omitted pulses:  n=18  rate=8.53+-1.72 Hz


## 5. Objective — global (continuous-stream) omission mismatch response

In [5]:
global_omission_mismatch_hz = float(all_standard_rates.mean() - all_omitted_rates.mean())
print(f"global omission mismatch (standard - omitted, continuous stream): {global_omission_mismatch_hz:+.2f} Hz")

# Computational diagnostic, not a claim of biological global-omission/
# predictive-coding validation -- a same-model firing-rate comparison
# between rhythm-consistent and rhythm-violating pulse events within a
# continuous stream, not a validated ERP/MMN result.
assert np.isfinite(global_omission_mismatch_hz)
assert len(all_omitted_rates) > 0, "no omission events sampled -- widen N_TRIALS or OMISSION_PROB"


global omission mismatch (standard - omitted, continuous stream): +5.15 Hz


## 6. Export

In [6]:
import json as _json
from pathlib import Path

OUT_DIR = Path("local/etude12")
OUT_DIR.mkdir(parents=True, exist_ok=True)

manifest = {
    "notebook": "jaxfne_etude_no_12_omission_global_coop",
    "jaxfne_version": jtfne.__version__,
    "paradigm": {
        "name": "omission_global_coop", "dur_ms": DUR_MS, "freq_hz": FREQ_HZ,
        "omission_prob": OMISSION_PROB, "n_trials": N_TRIALS,
        "stimulus_target": "L4_E_neurons", "n_stimulus_targets": len(l4e_idx),
    },
    "config": {"N_NEURONS": N_NEURONS, "DT_MS": DT_MS, "SEED": SEED},
    "results": {
        "n_standard_pulses": int(len(all_standard_rates)), "n_omitted_pulses": int(len(all_omitted_rates)),
        "standard_rate_hz_mean": float(all_standard_rates.mean()), "standard_rate_hz_std": float(all_standard_rates.std()),
        "omitted_rate_hz_mean": float(all_omitted_rates.mean()), "omitted_rate_hz_std": float(all_omitted_rates.std()),
        "global_omission_mismatch_hz": global_omission_mismatch_hz,
    },
}
(OUT_DIR / "manifest.json").write_text(_json.dumps(manifest, indent=2))
print("wrote", OUT_DIR / "manifest.json")


wrote local/etude12/manifest.json
